In [12]:
import pandas as pd
import numpy as np

df_modelo=pd.read_csv(
    r"C:\Users\param\OneDrive\Escritorio\Programacion\Soy Henry\Proyecto individual 1\MODELO\df_modelo")
    

In [13]:
print(df_modelo.dtypes)

budget               float64
original_language     object
overview              object
popularity           float64
revenue              float64
runtime              float64
title                 object
vote_average         float64
vote_count           float64
nombre_actor          object
nombre_director       object
titulo_pelicula       object
genero                object
pais_origen           object
release_año          float64
dtype: object


Se observa que tenemos numeros y texto. Hay que escalarlos en un solo vector...

In [14]:
import nltk
nltk.download('stopwords')  
nltk.download('wordnet')   
nltk.download('omw-1.4')   

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\param\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\param\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\param\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

Ahora falta normalizar el texto de todas las columnas que tienen un datatype object para que el cosine similarity pueda funcionar- agreguemos el overview que podrá ser de gran interes para recomendar peliculas al usar modelos de similitud de coseno.

In [27]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler


# Seleccion- se selecciona todas las columnas con dtype object
columnas_a_normalizar = ['original_language', 'overview',"title","nombre_actor","nombre_director","titulo_pelicula","genero","pais_origen"]

# Step 2: Normalizar- en ingles dado que el texto es en ingles.
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Funcion para reprocesar el texto
def preprocess_text(text):
    # Minusculas
    text = text.lower()
    
    # Quitar puntuación
    text = re.sub(r'[^\w\s]', '', text)
    
    # Quitar stop_word que esta en la biblioteca
    tokens = text.split()
    text = ' '.join([word for word in tokens if word not in stop_words])
    
    # Lemmatizar- llevar las palabras a la raiz
    tokens = text.split()
    text = ' '.join([lemmatizer.lemmatize(word) for word in tokens])
    
    return text

# Aplicar funcion a las columnas que tienen texto
for column in columnas_a_normalizar:
    df_modelo.loc[:, column] = df_modelo[column].apply(preprocess_text)

# Step 3: vectorizar el texto normalizado using TF-IDF
vectorizer = TfidfVectorizer()

# Combinar todo el texto de las columnas seleccionadas a una columna para vectorizar
text_data = df_modelo[columnas_a_normalizar].agg(' '.join, axis=1)
tfidf_matrix = vectorizer.fit_transform(text_data)

columnas_numericas = ['budget', 'popularity',"revenue","runtime","vote_average","vote_count","release_año"]
scaler = StandardScaler()
df_modelo[columnas_numericas] = scaler.fit_transform(df_modelo[columnas_numericas])

# Combinar vectores de texto a un formato denso
text_denso=tfidf_matrix.toarray()

#Convertir data numerica a un numpy array
numerical_data = df_modelo[columnas_numericas].values

#Combinar data- texto y numeros
data_combinada = np.hstack([numerical_data, text_denso])

sim_scores = cosine_similarity(data_combinada)

print(len(sim_scores))


4248


Creamos la función para recomendar peliculas-

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recomendacion(titulo):
    # Buscar el índice de la película
    titulo = titulo.lower()
    idx = df_modelo.index[df_modelo['title'].str.lower() == titulo].tolist()  # Case insensitive search
    if not idx:
        return "Película no encontrada"
    idx = idx[0]
    
    # Calcular la matriz de similitud del coseno entre todas las películas
    sim_scores = cosine_similarity(data_combinada)
    
    # Obtén los puntajes de similitud para la película seleccionada
    movie_similarities = sim_scores[idx]
    
    # Ordena las películas basadas en la similitud, excluyendo el índice de la película original
    sim_scores = list(enumerate(movie_similarities))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Obtener los índices de las películas más similares, excluyendo la misma película
    movie_indices = [i[0] for i in sim_scores if i[0] != idx][:10]  # 10 películas más similares

    # Devuelve los títulos de las películas más similares
    recomendaciones = df_modelo['title'].iloc[movie_indices].tolist()
    return {"Recomendaciones": recomendaciones}

# Prueba la función de recomendación
print(recomendacion("kizumonogatari part 1 tekketsu"))

{'Recomendaciones': ['kizumonogatari part 2 nekketsu', 'ghost shell arise border 5 pyrophoric cult', 'broken blade book six enclave lamentation', 'alone wilderness part ii', 'kizumonogatari part 3 reiketsu', 'ghost shell arise border 2 ghost whisper', 'monster high great scarrier reef', 'trailer park boy say goodnight bad guy', 'monster high boo york boo york', 'ghost shell arise border 1 ghost pain']}


Para probar peliculas random del dataset

In [35]:
#Traer un nombre aleatorio del dataset-
import random

titulo_aleatorio = df_modelo['title'].sample(n=1).values[0]

print(titulo_aleatorio)

kizumonogatari part 1 tekketsu
